In [ ]:
import os

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms.v2 as transforms

from lerobot.datasets.lerobot_dataset import LeRobotDataset


# ============================================================
# 1. CONFIG
# ============================================================

DATASET_ID = "lerobot/aloha_sim_transfer_cube_human"

SEED = 42
TRAIN_RATIO = 0.8

IMAGE_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 3
LR = 1e-3

# Keep small so it runs faster.
MAX_TRAIN = 3000
MAX_VAL = 800

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs("results", exist_ok=True)

print("Device:", DEVICE)


# ============================================================
# 2. EASY IMAGE TRANSFORM
# ============================================================

image_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
])


def preprocess_image(img):
    """
    Easy image preprocessing.

    Input:
        raw image from dataset

    Output:
        image tensor with shape [3, 64, 64]
        values between 0 and 1
    """
    img = image_transform(img)

    # Remove alpha channel if image has 4 channels.
    if img.shape[0] == 4:
        img = img[:3]

    # Convert grayscale to RGB.
    if img.shape[0] == 1:
        img = img.repeat(3, 1, 1)

    return img.cpu()


# ============================================================
# 3. SMALL HELPER FUNCTIONS
# ============================================================

def to_numpy(x):
    """Convert tensor to numpy."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)


def scalar(x):
    """Convert one-value tensor/array to a normal Python number."""
    arr = to_numpy(x)
    return arr.item() if arr.size == 1 else arr


def get_image_key(sample):
    """Find the image key in one dataset sample."""
    for key in sample.keys():
        if "image" in key.lower():
            return key
    return None


def get_task_id(sample):
    """Get task id if dataset has it. Otherwise use 0."""
    if "task_index" in sample:
        return int(scalar(sample["task_index"]))
    return 0


# ============================================================
# 4. LOAD DATASET
# ============================================================

print("\nLoading dataset...")

dataset = LeRobotDataset(DATASET_ID)
sample = dataset[0]

IMAGE_KEY = get_image_key(sample)

state_dim = to_numpy(sample["observation.state"]).shape[0]
action_dim = to_numpy(sample["action"]).shape[0]

print("Dataset loaded.")
print("Total timesteps:", len(dataset))
print("Image key:", IMAGE_KEY)
print("State dim:", state_dim)
print("Action dim:", action_dim)

print("\nSample keys:")
 for_key_list = list(sample.keys())
for key in for_key_list:
    print("-", key)


# ============================================================
# 5. EPISODE-BASED TRAIN / VAL SPLIT
# ============================================================

episode_info = dataset.episode_data_index

starts = to_numpy(episode_info["from"]).astype(int)
ends = to_numpy(episode_info["to"]).astype(int)

num_episodes = len(starts)

episodes = np.arange(num_episodes)
np.random.shuffle(episodes)

split_point = int(TRAIN_RATIO * num_episodes)

train_episodes = episodes[:split_point]
val_episodes = episodes[split_point:]


def get_indices_from_episodes(episode_list):
    """
    Turn episode numbers into timestep indices.

    This keeps train and validation split by episode.
    """
    indices = []

    for ep in episode_list:
        start = starts[ep]
        end = ends[ep]

        for idx in range(start, end):
            indices.append(idx)

    return np.array(indices)


train_indices = get_indices_from_episodes(train_episodes)
val_indices = get_indices_from_episodes(val_episodes)

# Keep the run small and fast.
train_indices = train_indices[:MAX_TRAIN]
val_indices = val_indices[:MAX_VAL]

print("\nEpisode split complete.")
print("Total episodes:", num_episodes)
print("Train episodes:", len(train_episodes))
print("Val episodes:", len(val_episodes))
print("Train examples used:", len(train_indices))
print("Val examples used:", len(val_indices))


# ============================================================
# 6. ACTION NORMALIZATION
# ============================================================

print("\nComputing action normalization...")

train_actions = []

for idx in train_indices:
    item = dataset[int(idx)]
    action = to_numpy(item["action"]).astype(np.float32)
    train_actions.append(action)

train_actions = np.stack(train_actions)

action_mean = train_actions.mean(axis=0)
action_std = train_actions.std(axis=0)

# Avoid division by zero.
action_std = np.maximum(action_std, 1e-6)

print("Action mean shape:", action_mean.shape)
print("Action std shape:", action_std.shape)


# ============================================================
# 7. TASK COUNT
# ============================================================

task_ids = []

for idx in train_indices:
    item = dataset[int(idx)]
    task_ids.append(get_task_id(item))

num_tasks = max(task_ids) + 1

print("Number of tasks:", num_tasks)


# ============================================================
# 8. PYTORCH DATASET
# ============================================================

class RobotDataset(Dataset):
    """
    Returns everything our 3 models may need:
    - state
    - image
    - task_id
    - normalized action
    - raw action
    """

    def __init__(self, dataset, indices, action_mean, action_std):
        self.dataset = dataset
        self.indices = indices

        self.action_mean = torch.tensor(action_mean, dtype=torch.float32)
        self.action_std = torch.tensor(action_std, dtype=torch.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        item = self.dataset[idx]

        state = torch.tensor(
            to_numpy(item["observation.state"]),
            dtype=torch.float32
        )

        action_raw = torch.tensor(
            to_numpy(item["action"]),
            dtype=torch.float32
        )

        action_norm = (action_raw - self.action_mean) / self.action_std

        output = {
            "state": state,
            "action_norm": action_norm,
            "action_raw": action_raw,
            "task_id": torch.tensor(get_task_id(item), dtype=torch.long),
        }

        if IMAGE_KEY is not None:
            output["image"] = preprocess_image(item[IMAGE_KEY])

        return output


train_ds = RobotDataset(dataset, train_indices, action_mean, action_std)
val_ds = RobotDataset(dataset, val_indices, action_mean, action_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))

print("\nBatch check:")
print("State:", batch["state"].shape)
print("Action:", batch["action_norm"].shape)

if IMAGE_KEY is not None:
    print("Image:", batch["image"].shape)

print("Task ID:", batch["task_id"].shape)


# ============================================================
# 9. MODEL 1: STATE-ONLY BC
# ============================================================

class StateOnlyBC(nn.Module):
    """
    robot_state -> action
    """

    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, action_dim),
        )

    def forward(self, batch):
        state = batch["state"].to(DEVICE)
        return self.model(state)


# ============================================================
# 10. MODEL 2: IMAGE + STATE BC
# ============================================================

class ImageStateBC(nn.Module):
    """
    image + robot_state -> action
    """

    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.image_model = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),

            nn.Flatten(),

            # 64x64 -> 32x32 -> 16x16
            nn.Linear(32 * 16 * 16, 128),
            nn.ReLU(),
        )

        self.state_model = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(128 + 64, 128),
            nn.ReLU(),

            nn.Linear(128, action_dim),
        )

    def forward(self, batch):
        image = batch["image"].to(DEVICE)
        state = batch["state"].to(DEVICE)

        image_features = self.image_model(image)
        state_features = self.state_model(state)

        combined = torch.cat([image_features, state_features], dim=1)

        return self.head(combined)


# ============================================================
# 11. MODEL 3: TASK-CONDITIONED BC
# ============================================================

class TaskConditionedBC(nn.Module):
    """
    image + robot_state + task_id -> action
    """

    def __init__(self, state_dim, action_dim, num_tasks):
        super().__init__()

        self.image_model = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),

            nn.Flatten(),

            nn.Linear(32 * 16 * 16, 128),
            nn.ReLU(),
        )

        self.state_model = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
        )

        self.task_embedding = nn.Embedding(num_tasks, 16)

        self.head = nn.Sequential(
            nn.Linear(128 + 64 + 16, 128),
            nn.ReLU(),

            nn.Linear(128, action_dim),
        )

    def forward(self, batch):
        image = batch["image"].to(DEVICE)
        state = batch["state"].to(DEVICE)
        task_id = batch["task_id"].to(DEVICE)

        image_features = self.image_model(image)
        state_features = self.state_model(state)
        task_features = self.task_embedding(task_id)

        combined = torch.cat(
            [image_features, state_features, task_features],
            dim=1
        )

        return self.head(combined)


# ============================================================
# 12. TRAINING FUNCTION
# ============================================================

def train_model(model, model_name):
    model = model.to(DEVICE)

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("\n" + "=" * 60)
    print("Training:", model_name)
    print("=" * 60)

    for epoch in range(1, EPOCHS + 1):
        model.train()

        train_loss_total = 0
        train_count = 0

        for batch in train_loader:
            target = batch["action_norm"].to(DEVICE)

            pred = model(batch)

            loss = loss_fn(pred, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * target.size(0)
            train_count += target.size(0)

        train_loss = train_loss_total / train_count

        val_loss, raw_mae = evaluate_model(model, loss_fn)

        print(
            f"Epoch {epoch} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Raw Val MAE: {raw_mae:.4f}"
        )

    return val_loss, raw_mae


# ============================================================
# 13. EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, loss_fn):
    model.eval()

    val_loss_total = 0
    val_count = 0

    all_preds_raw = []
    all_true_raw = []

    mean = torch.tensor(action_mean, dtype=torch.float32).to(DEVICE)
    std = torch.tensor(action_std, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        for batch in val_loader:
            target_norm = batch["action_norm"].to(DEVICE)
            target_raw = batch["action_raw"].to(DEVICE)

            pred_norm = model(batch)

            loss = loss_fn(pred_norm, target_norm)

            # Convert prediction back to real action scale.
            pred_raw = pred_norm * std + mean

            val_loss_total += loss.item() * target_norm.size(0)
            val_count += target_norm.size(0)

            all_preds_raw.append(pred_raw.cpu())
            all_true_raw.append(target_raw.cpu())

    preds = torch.cat(all_preds_raw).numpy()
    true = torch.cat(all_true_raw).numpy()

    raw_mae = np.mean(np.abs(preds - true))
    val_loss = val_loss_total / val_count

    return val_loss, raw_mae


# ============================================================
# 14. RUN ALL MODELS
# ============================================================

results = []

# Model 1: state only
state_model = StateOnlyBC(state_dim, action_dim)

val_loss, raw_mae = train_model(
    model=state_model,
    model_name="state_only_bc"
)

results.append({
    "model": "state_only_bc",
    "val_loss": val_loss,
    "raw_val_mae": raw_mae,
})


# Model 2 and 3 need images.
if IMAGE_KEY is not None:
    image_state_model = ImageStateBC(state_dim, action_dim)

    val_loss, raw_mae = train_model(
        model=image_state_model,
        model_name="image_state_bc"
    )

    results.append({
        "model": "image_state_bc",
        "val_loss": val_loss,
        "raw_val_mae": raw_mae,
    })


    task_model = TaskConditionedBC(state_dim, action_dim, num_tasks)

    val_loss, raw_mae = train_model(
        model=task_model,
        model_name="task_conditioned_bc"
    )

    results.append({
        "model": "task_conditioned_bc",
        "val_loss": val_loss,
        "raw_val_mae": raw_mae,
    })

else:
    print("No image key found, so image models were skipped.")


# ============================================================
# 15. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

print("\nFinal model comparison:")
print(results_df)

results_df.to_csv("results/bc_model_comparison.csv", index=False)

print("\nSaved results to:")
print("results/bc_model_comparison.csv")